# Diarization benchmark — pyannote.audio 3.1 vs Gemma 4 (E2B / E4B / 12B)

Bootstrap only (Phase 01). Runs **both** in Colab and locally — every cell branches on
`IN_COLAB`, so the same notebook can be prototyped locally and then run unchanged on a T4.

- **Colab:** needs a secret named `HF_TOKEN` (notebook access enabled), with terms accepted for
  `pyannote/speaker-diarization-3.1` and the `google/gemma-4-*-it` variants.
- **Local:** `uv sync --extra dev && uv run jupyter lab notebooks/benchmark.ipynb`, with
  `HF_TOKEN` exported in the shell.

In [ ]:
import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
REPO_URL = "https://github.com/Kushal-0532/capstone_diarization_benchmark.git"

if IN_COLAB:
    REPO_DIR = Path("/content/capstone_diarization_benchmark")
    cmd = ["git", "-C", str(REPO_DIR), "pull", "--ff-only"] if REPO_DIR.is_dir() else ["git", "clone", REPO_URL, str(REPO_DIR)]
    subprocess.run(cmd, check=True)
else:
    REPO_DIR = next(p for p in Path.cwd().parents if (p / "pyproject.toml").is_file())
print("IN_COLAB", IN_COLAB, "|", REPO_DIR)

In [ ]:
# Colab ships a CUDA-matched torch; pyproject deliberately does not pin over it.
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)

# An editable install writes a .pth that only a fresh interpreter reads, so importing right
# after installing fails without a kernel restart. Flat layout means the path insert is enough.
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import benchmark
print("benchmark", benchmark.__version__, "from", benchmark.__file__)

In [ ]:
from benchmark import config

if IN_COLAB:
    from google.colab import drive

    drive.mount(str(config.DRIVE_MOUNT))
    RESULTS_DIR = config.DRIVE_RESULTS_DIR
else:
    RESULTS_DIR = config.RESULTS_DIR
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("results ->", RESULTS_DIR)

In [ ]:
# Never print the token, never hardcode it (C4).
if IN_COLAB:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

assert os.environ.get("HF_TOKEN"), "HF_TOKEN missing: Colab secret, or export it locally"
print("HF_TOKEN loaded")

In [ ]:
from benchmark import env

env.print_description()